# <center>Transporte óptimo</center>

In [9]:
#!pip install kaleido
#!pip install pulp

import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm, multivariate_normal
from plotly.subplots import make_subplots
import pulp

In [10]:
rut = 19_476_309_3
np.random.seed(rut)

### Pregunta 1

In [11]:
def kantorovich_problem(distr1: dict, distr2: dict, cost: callable):
    '''
    Resuelve el problema de transporte óptimo para dos distribuciones discretas 
    usando como costo de transporte el cuadrado de la distancia (C_ij = ||x_i - x_j||^2).

    Parameters:
    - distr1: {'loc': list, 'mass': list} distribución de origen (ubicación de los puntos y masas).
    - distr1: {'loc': list, 'mass': list} distribución de destinto (ubicación de los puntos y masas).
    - cost: función que evalúa el costo de transporte entre un punto de distr1 y
        otro de distr2.
    
    Returns:
    - T: matriz de transporte óptimo.
    '''

    n, m = len(distr1['loc']), len(distr2['loc'])
    model = pulp.LpProblem(sense=pulp.LpMinimize)

    # Variables del problema:
    P = [[pulp.LpVariable(f'p_{i}{j}', lowBound=0)
          for j in range(m)] for i in range(n)]

    # Función objetivo:
    model += pulp.lpSum([
        P[i][j] * cost(xi, yj) 
        for i, xi in enumerate(distr1['loc'])
        for j, yj in enumerate(distr2['loc'])
    ])

    # Restricciones:
    for i, xi in enumerate(distr1['mass']):
        model += pulp.lpSum(P[i][j] for j in range(m)) == xi
        
    for j, yj in enumerate(distr2['mass']):
        model += pulp.lpSum(P[i][j] for i in range(n)) == yj
    
    # Matriz de transporte óptimo:
    model.solve(pulp.PULP_CBC_CMD(msg=0))
    P_optimal = np.array([[P[i][j].varValue for j in range(m)] for i in range(n)])

    return P_optimal


def transportation_cost(P, distr1: dict, distr2: dict, cost: callable):
    '''Calcula el costo total de transporte entre dos distribuciones dada una matriz de
    transporte P y una función de costos.'''

    cost = sum(
        P[i][j] * cost(xi, yj)
        for i, xi in enumerate(distr1['loc'])
        for j, yj in enumerate(distr2['loc'])
    )
    return cost

### Pregunta 3

In [12]:
def quadratic_cost(x: np.array, y: np.array):
    '''Retorna el cuadrado de la distancia entre dos puntos.'''
    return np.linalg.norm(x - y) ** 2

def analytical_gaussian_optimal_transport(mean1: np.array, mean2: np.array, var1: np.array, var2: np.array):
    '''Retorna la función de transporte óptimo para dos distribuciones gaussianas. Esta evalúa un punto de la
    distribución de origen y entrega su punto de llegada en la distribución 2.'''

    def T(x):
        var1_sqrt = np.sqrt(var1)
        var1_sqrt_inverse = 1/var1_sqrt if var1_sqrt.size == 1 else np.linalg.inv(var1_sqrt)
        A = var1_sqrt_inverse * np.sqrt(var1_sqrt * var2 * var1_sqrt) * var1_sqrt_inverse
        return mean2 + A * (x - mean1) if var1_sqrt.size == 1 else mean2 + A @ (x - mean1)
    
    return T

def analytical_gaussian_wasserstein(mean1: np.array, mean2: np.array, var1: np.array, var2: np.array):
    '''Retorna el costo de transporte óptimo para dos distribuciones gaussianas.'''
    var1_sqrt = np.sqrt(var1)
    aux = var1 + var2 - 2 * var1_sqrt * var2 * var1_sqrt
    bures_metric = aux if var1_sqrt.size == 1 else np.trace(aux)
    return quadratic_cost(mean1, mean2) + bures_metric

In [13]:
# Parámetros fijos:
mean1, var1 = 0.0, 1.0  # X1.
mean2, var2 = 4.0, 0.5  # X2.
n, m = 10, 15

# Muestras:
x1_samples = np.random.normal(mean1, var1, n)
x2_samples = np.random.normal(mean2, var2, m)

# Histograma de las muestras:
common_range = np.linspace(min(min(x1_samples), min(x2_samples)) - 1, max(max(x1_samples), max(x2_samples)) + 1, 400)
gaussian_x = norm.pdf(common_range, 0, 1)
gaussian_y = norm.pdf(common_range, 4, 0.5)

fig = go.Figure()
fig.add_trace(go.Histogram(x=x1_samples, histnorm='probability density', nbinsx=20, opacity=0.8))
fig.add_trace(go.Histogram(x=x2_samples, histnorm='probability density', nbinsx=20, opacity=0.8))
fig.add_trace(go.Scatter(x=common_range, y=gaussian_x, mode='lines'))
fig.add_trace(go.Scatter(x=common_range, y=gaussian_y, mode='lines'))
fig.update_layout(showlegend=False, xaxis_title='x', yaxis_title='density', margin=dict(t=10, b=10, l=10, r=10))
fig.write_image('imgs/p3.3.1.pdf', width=800, height=350)
fig.show()

In [14]:
# Medidas discretas (de masa uniforme):
distr1 = {'loc': x1_samples, 'mass': [1.0/n] * n}
distr2 = {'loc': x2_samples, 'mass': [1.0/m] * m}

# Transporte óptimo exacto:
analytical_optimal_operator = analytical_gaussian_optimal_transport(mean1, mean2, var1, var2)
analytical_optimal_cost = analytical_gaussian_wasserstein(mean1, mean2, var1, var2)
print(f'Costo mínimo teórico: {analytical_optimal_cost:.4f}')

# Transporte óptimo numérico:
P_numerical = kantorovich_problem(distr1, distr2, quadratic_cost)
numerical_optimal_cost = transportation_cost(P_numerical, distr1, distr2, quadratic_cost)
print(f'Costo mínimo numérico: {numerical_optimal_cost:.4f}')

# Gráfico de transporte óptimo:
x_discrete = distr1['loc']
y_discrete = [np.average(distr2['loc'], weights=P_numerical[i]) for i in range(n)]
x_continuous = np.linspace(min(x_discrete), max(x_discrete), 100)
y_continuous = analytical_optimal_operator(x_continuous)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_continuous, y=y_continuous, mode='lines'))
fig.add_trace(go.Scatter(x=x_discrete, y=y_discrete, mode='markers'))
fig.update_layout(xaxis_title='Input distribution', yaxis_title='Target distribution', showlegend=False, margin=dict(t=10, b=10, l=10, r=10))
fig.write_image('imgs/p3.3.2.pdf', width=800, height=350)
fig.show()

Costo mínimo teórico: 16.5000
Costo mínimo numérico: 17.5517


### Pregunta 4

In [15]:
# Parámetros fijos:
mu1, cov1 = np.array([0, 0]), np.array([[1.0, 0.0], [0.0, 1.0]])
mu2, cov2 = np.array([3, 5]), np.array([[0.4, 0.2], [0.2, 0.3]])
n, m = 10, 15

# Muestras:
x1_samples = np.random.multivariate_normal(mu1, cov1, n)
x2_samples = np.random.multivariate_normal(mu2, cov2, m)

# Gráfico:
fig = make_subplots(rows=1, cols=2, subplot_titles=('X1', 'X2'))

for mean, covariance, samples, i in zip([mu1, mu2], [cov1, cov2], [x1_samples, x2_samples], [1, 2]):
    range_factor = 4 * covariance[i-1, i-1]
    x_contour = np.linspace(mean[0] - range_factor, mean[0] + range_factor, 100)
    y_contour = np.linspace(mean[1] - range_factor, mean[1] + range_factor, 100)
    
    grid = np.meshgrid(x_contour, y_contour)
    p = multivariate_normal(mean, covariance).pdf(np.dstack(grid))

    fig.add_trace(go.Contour(z=p, x=x_contour, y=y_contour, colorscale='Viridis', showscale=False), row=1, col=i)
    fig.add_trace(go.Scatter(x=samples[:, 0], y=samples[:, 1], mode='markers', marker=dict(size=10, color='red')), row=1, col=i)
    fig.update_xaxes(title_text=f'1º coordinate', row=1, col=i)
    fig.update_yaxes(title_text=f'2º coordinate', row=1, col=i)

fig.update_layout(showlegend=False, margin=dict(t=30, b=10, l=10, r=10))
fig.write_image('imgs/p3.4.1.pdf', width=800, height=350)
fig.show()


In [16]:
# Medidas discretas (de masa uniforme):
distr1 = {'loc': x1_samples, 'mass': [1.0/n] * n}
distr2 = {'loc': x2_samples, 'mass': [1.0/m] * m}

# Transporte óptimo numérico:
P_numerical = kantorovich_problem(distr1, distr2, quadratic_cost)

# Gráfico:
fig = make_subplots(rows=1, cols=2)
for dim in (0, 1):
    x_discrete = distr1['loc'][:, dim]
    y_discrete = [np.average(distr2['loc'][:, dim], weights=P_numerical[i]) for i in range(n)]
    fig.add_trace(go.Scatter(x=x_discrete, y=y_discrete, mode='markers', name='Transporte óptimo (discreto) esperado'), row=1, col=dim+1)
    fig.update_xaxes(title_text=f'Input distribution ({dim+1}º coordinate)', row=1, col=dim+1)
    fig.update_yaxes(title_text=f'Target distribution ({dim+1}º coordinate)', row=1, col=dim+1)
fig.update_layout(showlegend=False, margin=dict(t=10, b=10, l=10, r=10))
fig.write_image('imgs/p3.4.2.pdf', width=800, height=350)
fig.show()